# Lab 5 — Supervisor + Deploy to AgentCore Runtime

**What this lab is.** We bring everything together. The **Supervisor** is the "brain" that
coordinates all the specialists: it runs the safety checks, retrieves approved information,
escalates clinical questions, and verifies the answer before replying. Then we **deploy** it
to **AgentCore Runtime** — a managed, always-on home in AWS — so it can be called like a
service.

**Why we do it.** Up to now each piece worked on its own. The Supervisor is what makes them
work *together* as one safe assistant, with budgets (limits on steps/time/cost) so it can't
run away. Deploying it means it's no longer just running in a notebook — it's a real endpoint.

**Why it's needed here.** A patient's question often has several parts (a safe part, a
clinical part). The Supervisor is what handles that correctly: answer the safe part from
documents, escalate the clinical part, and verify before sending anything.

**How it helps the project.** After this lab, CareConnect exists as a deployed service that
the API and frontend (lab-06) can call.

**The use case.** "How do I prep for my colonoscopy, can I get my refill in time, and should
I stop my metformin?" → the Supervisor answers prep from documents, stages the refill, and
**escalates** the "should I stop my metformin" part to a clinician — all in one reply.

---

## Prerequisite — run these two setup cells first (every lab has them)

Before this lab's own steps, run the **two setup cells** below. Every notebook (lab-00
through lab-08) starts with these same two cells — on purpose, not by mistake.

**Why they repeat in every notebook.** Each notebook runs in its own fresh "kernel"
(a separate Python session) with no memory of the other notebooks. So each one has to set
itself up from scratch. These two cells are that setup.

**Cell 1 — "bootstrap":** finds the project's main folder (the one containing `lab_helpers`)
no matter where the notebook is opened from, so `import lab_helpers...` always works.

**Cell 2 — "preflight":** checks all helper files are present before the lab begins, and stops
with a clear message if anything is missing — instead of failing confusingly later.

**Do I run them?** Yes — run both, in order, at the top of **every** lab. They take a second
and prevent the most common setup problems. After these two, continue with the lab's steps.

In [1]:
# WHAT THIS CELL DOES (plain English):
# - It looks at the current folder, then its parent, then its parent's parent, and so on,
#   until it finds the folder that contains "lab_helpers". That folder is our project root.
# - It then switches into that folder and adds it to Python's search path, so that
#   'import lab_helpers...' works from anywhere.
# - If it never finds "lab_helpers", it stops with a clear message instead of a confusing error later.
# You do not need to edit anything here — just run it first.
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

Repo root: /home/sagemaker-user/careconnect-patient-assistant-k21


In [2]:
# WHAT THIS CELL DOES (plain English):
# - It checks that each helper file we rely on is actually present on disk.
# - If any are missing, it stops now with a clear list of what to upload, rather than
#   failing in the middle of the lab.
# Run this straight after the bootstrap cell above.
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

Preflight OK — all helper files present.


### Step 1 — Test the Supervisor locally (before deploying)

**What:** run the Supervisor's logic right here in the notebook against two questions.

**Why:** it's much faster to catch problems locally than after deploying. We test a simple
question and the tricky multi-part metformin question to confirm the clinical part gets
escalated, not answered.

**Note on `nest_asyncio`:** Jupyter already runs a background event loop, so we use
`nest_asyncio` to let us call the Supervisor's async code from a notebook cell. This is a
notebook-only convenience and has nothing to do with the agent's behaviour.

In [4]:
# WHAT THIS CELL DOES (plain English):
# - 'nest_asyncio' lets us run the Supervisor's asynchronous code inside Jupyter (which
#   already runs its own event loop). Without it you'd get "event loop is already running".
# - We reload the Supervisor code, then ask it two questions and print the answers:
#     * a simple visiting-hours question (answered from documents)
#     * the multi-part metformin question (the clinical part must be ESCALATED, not answered).
import nest_asyncio, asyncio
nest_asyncio.apply()   # allows asyncio.run() inside Jupyter's running loop

import importlib, lab_helpers.runtime_entrypoint as rt
importlib.reload(rt)

async def ask(q):
    return await rt.invoke({"prompt": q})

print(asyncio.run(ask("What are the visiting hours at Riverside Health?")))
print("---")
print(asyncio.run(ask(
    "I have a colonoscopy next Tuesday and I am almost out of my metformin. "
    "How do I prepare, can I get my refill in time, and should I stop taking it?")))

I can help with approved Riverside Health information, but I cannot provide medical diagnosis, medication dosage changes, treatment recommendations, or medical triage. Please contact a qualified healthcare professional for medical advice.
---
I can help with approved Riverside Health information, but I cannot provide medical diagnosis, medication dosage changes, treatment recommendations, or medical triage. Please contact a qualified healthcare professional for medical advice.


### Step 2 — Deploy the Supervisor to AgentCore Runtime

**What:** package the Supervisor and deploy it to AgentCore Runtime using the starter
toolkit — all from Python, no separate command-line tool needed.

**Why:** deploying turns our code into a managed, scalable endpoint that runs even when the
notebook is closed. This is what the API and frontend will call. The toolkit builds a
container, pushes it, and creates the runtime for us.

In [5]:
# WHAT THIS CELL DOES (plain English):
# - Creates the permission role the deployed Supervisor will run with.
# - 'runtime.configure(...)' tells the toolkit which file to deploy (our Supervisor
#   entrypoint), which role to use, and to auto-create the container registry.
# - This cell prepares the deployment; the next cell actually launches it.
import boto3
from bedrock_agentcore_starter_toolkit import Runtime
import lab_helpers.utils as u

exec_role = u.create_agentcore_runtime_execution_role()
runtime = Runtime()

runtime.configure(
    entrypoint="lab_helpers/runtime_entrypoint.py",
    execution_role=exec_role,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=u.REGION,
    agent_name=u.RUNTIME_AGENT_NAME,
)
print("Configured runtime.")

Created role CareConnectRuntimeRole-sdk


Entrypoint parsed: file=/home/sagemaker-user/careconnect-patient-assistant-k21/lab_helpers/runtime_entrypoint.py, bedrock_agentcore_name=runtime_entrypoint
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: careconnect_supervisor_sdk


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Generated Dockerfile: /home/sagemaker-user/careconnect-patient-assistant-k21/Dockerfile

Generated .dockerignore: /home/sagemaker-user/careconnect-patient-assistant-k21/.dockerignore
Setting 'careconnect_supervisor_sdk' as default agent
Bedrock AgentCore configured: /home/sagemaker-user/careconnect-patient-assistant-k21/.bedrock_agentcore.yaml


Configured runtime.


In [6]:
# WHAT THIS CELL DOES (plain English):
# - Actually builds and deploys the Supervisor to AgentCore Runtime (this can take a few minutes).
# - Prints the deployed agent's ARN (its unique address) and saves it to Parameter Store so the
#   API in lab-06 can find it.
launch_result = runtime.launch()
print("Launched:", launch_result.agent_arn)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn", launch_result.agent_arn)

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'careconnect_supervisor_sdk' to account 831963379350 (us-east-1)
Generated image tag: 20260907-141313-329
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: careconnect_supervisor_sdk


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-careconnect_supervisor_sdk


ECR repository available: 831963379350.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-careconnect_supervisor_sdk
Using execution role from config: arn:aws:iam::831963379350:role/CareConnectRuntimeRole-sdk
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: careconnect_supervisor_sdk
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-66d6629e21
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-66d6629e21
Creating IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-66d6629e21
✓ Role created: arn:aws:iam::831963379350:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-66d6629e21
Attaching inline policy: CodeBuildExecutionPolicy to role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-66d6629e21
✓ Policy attached: CodeBuildExecutionPolicy
Waiting for IAM role propagation...
CodeBuild execution role creation complete: arn:aws:iam::831963379350:role/AmazonBedrockAgentCoreSDKCodeBuild-us

Launched: arn:aws:bedrock-agentcore:us-east-1:831963379350:runtime/careconnect_supervisor_sdk-qrzEn6Dt3d


In [7]:
# WHAT THIS CELL DOES (plain English):
# - Waits in a loop, checking every 20 seconds, until the deployed runtime reports READY/ACTIVE.
# - You must wait for READY before trying to call it in the next cell.
import time
while True:
    st = runtime.status()
    ep = getattr(st, "endpoint", None)
    print("status:", ep.get("status") if ep else "provisioning...")
    if ep and ep.get("status") in ("READY", "ACTIVE"):
        break
    time.sleep(20)

Retrieved Bedrock AgentCore status for: careconnect_supervisor_sdk


status: READY


### Step 3: Invoke the deployed runtime

In [8]:
# WHAT THIS CELL DOES (plain English):
# - Calls the deployed Supervisor endpoint with a test question and prints its reply.
# - This confirms the deployment works and is reachable as a real service.
import json
agentcore = boto3.client("bedrock-agentcore", region_name=u.REGION)
arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn")
resp = agentcore.invoke_agent_runtime(
    agentRuntimeArn=arn,
    payload=json.dumps({"prompt": "How should I prepare for my colonoscopy?"}).encode())
print(resp["response"].read().decode())

"Approved Riverside Health information:\nPassage 1\n# Upper Endoscopy (Gastroscopy) Preparation (Riverside Health — Approved Leaflet, v2026-07) Follow any specific instructions from your care team, which take priority over this general leaflet. ## Fasting Do not eat solid food for at least 6 hours before the procedure. You may usually have small sips of water up to 2 hours before, unless your care team tells you otherwise. ## The day of the procedure Wear comfortable clothing. Leave jewellery and valuables at home. Bring your photo ID and insurance card, and arrive 30 minutes early. ## Sedation and going home If you receive sedation, arrange for a responsible adult to take you home. Do not drive, operate machinery, or sign legal documents for the rest of the day. ## Dentures, glasses, contact lenses You may be asked to remove dentures, glasses, or contact lenses before the procedure. Bring a case for them. ## Medication and clinical questions Do not start, stop, or change any medicatio

## Lab 5 complete ✅

Supervisor deployed to a managed AgentCore Runtime endpoint, invoked from the SDK.